In [1]:
import pandas as pd
from tqdm import tqdm
import time
import numpy as np

from __future__ import annotations
from yandex_cloud_ml_sdk import YCloudML


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", None)

In [2]:
X_test = pd.read_csv('ver_1/X_test.csv',index_col=0)

In [3]:
X_test.head(10)

,purpose
222196,Благотворительное пожертвование на уставную деятельность. НДС не облагается
474146,БЛАГОТВОРИТЕЛЬНЫЙ ВЗНОС ЗА 13/04/2025;Добровольное пожертвование САВЕЛЬЕВ ДМИТРИЙ АЛЕКСАНДРОВИЧ;semenixina.mar@yandex.ru
233469,"Перевод средств по договору б/н от 23.07.2020 по Реестру Операций от 25.07.2024. Сумма комиссии 8 руб. 70 коп., НДС не облагается."
7601,"Оплата ветеринарных услуг по счету 11369 от 14.03.2023 (Ярулина, соб Лохматая), НДС не облагается"
436337,Добровольное пожертвование 10.00
282842,Благотворительное пожертвование на уставную деятельность. НДС не облагается
338292,(283.0703.0120165300.635 л/с 02133D07240) ДК 24В26 л/с 03283D10001 БО 213D100024000012 Субсид на иные цели (фин обеспеч испол мун соц заказа на оказан мун услуг) по соглаш 13 от 09.01.2024 пр1139 от 12.11.2024
560686,Благотворительное пожертвование на уставную деятельность. НДС не облагается
393765,Благотворительное пожертвование на уставную деятельность. НДС не облагается
207682,Благотворительное пожертвование на уставную деятельность. НДС не облагается


In [4]:
y_pred_ygpt = pd.DataFrame(columns=["universal_category"])

sdk = YCloudML(
        folder_id="YOUR_FOLDER_ID",
        auth="YOUR_YANDEX_CLOUD_TOKEN",
    )

task_description="""Представь, что ты обученная модель для классификации благотворитрельных платежей по нескольким универсальным категориям доходных статей.
                    Есть датасет с тренировочными данными - X_train.csv и y_train.csv – с целевым признаком для классификации по нескольким категориям, изучи внимательно тренировочные данные и далее я буду давать тебе тестовые строки - нужно предложить одну из искомых категорий статей клаccификации из общего набора:
                    пожертвования от физических лиц (напрямую)
                    пожертвования через платформы
                    продажа товаров
                    прочие недоходные операции
                    продажа услуг
                    пожертвования от юридических лиц (напрямую)
                    финансовые доходы
                    членские и учредительские взносы
                    гранты субсидии конкурсы
                    Обрати внимание, что в тренировочных данных миксплат, юмани относятся к категориям не платформ а частных пожертвований, потому что это технические способы сбора средств частных пожертвований. Платформы - это специальные благотворительные программы, сайты, заточенные именно сбор целевых или благотворительных платежей, а миксплат, юмани, различные сайты с наличием слова "деньги" в названиях - это обычные платежные системы.
                    Важно, например тбанк и тбанк благотворительность - это очень разные сферы, первое - это просто финансовый канал и сам по себе он относится к поступлениям от физ. лиц, а второе в это именно благотворительная платформа тбанка – смотри внимательно и соотноси названия статей и универсальные категории в тренировочной выборке.
                    Если будет необходимо - сходи в поиск и проверь названия в интернете
                    Как будешь готов предложи универсальные категории для платежей из X_test.csv и верни назад список в формате csv с двумя колонками - id соответствующий платежу в X_test и твой прогноз по категории. Не пропусти ни одного платежа - точно каждому назначь универсальную категорию, приложи все усилия, и будь очень внимателен к редким классам - данные по ним постарались выложить максимально полные в тренировочной выборке.
                    действуй строго по списку выше, не придумывай дополнительных категорий и дай ответы ко всем платежам из X_test.csv - строго по их id, проверяя не дублируются ли id и не перепутай входные тексты""".replace("\n", " ").strip()
labels = [
    "пожертвования от физических лиц (напрямую)",
    "пожертвования через платформы",
    "продажа товаров",
    "прочие недоходные операции",
    "продажа услуг",
    "пожертвования от юридических лиц (напрямую)",
    "финансовые доходы",
    "членские и учредительские взносы",
    "гранты субсидии конкурсы"
]

import numpy as np
import time
from tqdm import tqdm

# Создаём модель один раз
model = sdk.models.text_classifiers("yandexgpt").configure(
    task_description=task_description,
    labels=labels,
)

for index_, text in tqdm(X_test['purpose'].items(), total=len(X_test)):
    attempts = 0
    max_attempts = 3
    while attempts < max_attempts:
        try:
            result = model.run(str(text))  # приводим текст к строке
            best_label = max(result, key=lambda x: x.confidence).label
            y_pred_ygpt.loc[index_, 'universal_category'] = best_label
            break
        except Exception as e:
            attempts += 1
            print(f"Ошибка на index {index_}, попытка {attempts}/{max_attempts}: {e}")
            time.sleep(2)
    else:
        # если все попытки неудачные, присваиваем NaN
        y_pred_ygpt.loc[index_, 'universal_category'] = np.nan

y_pred_ygpt.to_csv("y_pred_ygpt.csv", index=True, header=["universal_category"])
    

100%|██████████| 5000/5000 [2:50:46<00:00,  2.05s/it]  


In [5]:
y_pred_ygpt.to_csv("y_pred_ygpt.csv", index=True, header=["universal_category"])